In [253]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm

## Cleaning current DEM data (will be updated soon once proper data is acquired)

In [254]:
mjf_dem_raw_1 = pd.read_csv("raw DEM data/EP NO2 Aug+Sep 2025.csv")
mjf_dem_raw_2 = pd.read_csv("raw DEM data/MJF DEM cleaned.csv")

In [255]:
mjf_dem_raw_1.head()

,Date,NO2
0,8/1/25 3:00,1.4
1,8/1/25 4:00,1.3
2,8/1/25 5:00,2.1
3,8/1/25 6:00,2.7
4,8/1/25 7:00,3.5


In [256]:
mjf_dem_raw_2.head()

,datetime_est,no2,o3
0,2025-10-12 00:00:00,2.3,19.0
1,2025-10-12 01:00:00,2.0,21.2
2,2025-10-12 02:00:00,2.3,21.0
3,2025-10-12 03:00:00,1.7,23.0
4,2025-10-12 04:00:00,1.7,23.7


In [257]:
mjf_dem_raw_1 = mjf_dem_raw_1.rename(columns={"Date": "datetime_est", "NO2": "no2"})

#use mask to isolate midnight times, which by default do not have a 00:00:00 attached
mask = ~mjf_dem_raw_1["datetime_est"].str.contains(":", na=False)
mjf_dem_raw_1.loc[mask, "datetime_est"] = (mjf_dem_raw_1.loc[mask, "datetime_est"] + " 0:00")

#add empty o3 column for later merging
mjf_dem_raw_1["o3"] = pd.NA

mjf_dem_raw_1["datetime_est"] = pd.to_datetime(mjf_dem_raw_1["datetime_est"], format="%m/%d/%y %H:%M")

mjf_dem_raw_1.head(24)

,datetime_est,no2,o3
0,2025-08-01 03:00:00,1.4,<NA>
1,2025-08-01 04:00:00,1.3,<NA>
2,2025-08-01 05:00:00,2.1,<NA>
3,2025-08-01 06:00:00,2.7,<NA>
4,2025-08-01 07:00:00,3.5,<NA>
5,2025-08-01 08:00:00,2.9,<NA>
6,2025-08-01 09:00:00,1.6,<NA>
7,2025-08-01 10:00:00,1.2,<NA>
8,2025-08-01 11:00:00,1.3,<NA>
9,2025-08-01 12:00:00,1.0,<NA>


In [258]:
mjf_dem_raw_2["datetime_est"] = pd.to_datetime(mjf_dem_raw_2["datetime_est"])

mjf_dem_cleaned = pd.concat([mjf_dem_raw_1, mjf_dem_raw_2], ignore_index=True).sort_values("datetime_est")
mjf_dem_cleaned = mjf_dem_cleaned.reset_index(drop=True)

mjf_dem_cleaned.head(30)
mjf_dem_cleaned.to_csv("cleaned DEM data/dem_cleaned.csv", index=False)

/var/folders/cv/16s476bd69z2kjqnmm06ls5c0000gn/T/ipykernel_59832/849428465.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  mjf_dem_cleaned = pd.concat([mjf_dem_raw_1, mjf_dem_raw_2], ignore_index=True).sort_values("datetime_est")


## Cleaning quant data

In [259]:
dpw_raw = pd.read_csv("raw Quant data/dpw raw.csv")
mjf_raw = pd.read_csv("raw Quant data/mjf raw.csv")
mjf_recal = pd.read_csv("raw Quant data/mjf recalibrated.csv")
pema_raw = pd.read_csv("raw Quant data/pema raw.csv")
pha_raw = pd.read_csv("raw Quant data/pha raw.csv")

In [260]:
dpw_raw = (dpw_raw[["timestamp_local", "no2", "o3"]].rename(columns={"timestamp_local": "datetime_est"}))
dpw_raw["datetime_est"] = pd.to_datetime(dpw_raw["datetime_est"])

dpw_raw.head(30)

,datetime_est,no2,o3
0,2026-05-31 19:59:45+00:00,25.082,28.501
1,2026-05-31 19:58:45+00:00,24.654,29.081
2,2026-05-31 19:57:45+00:00,23.764,28.863
3,2026-05-31 19:56:45+00:00,22.344,28.328
4,2026-05-31 19:55:45+00:00,22.344,29.697
5,2026-05-31 19:54:45+00:00,25.490,27.914
6,2026-05-31 19:53:45+00:00,26.753,27.545
7,2026-05-31 19:52:45+00:00,25.143,29.443
8,2026-05-31 19:51:45+00:00,24.745,28.669
9,2026-05-31 19:50:45+00:00,25.224,29.005


In [261]:
dpw_quant_cleaned = (dpw_raw.set_index("datetime_est").resample("h").mean().reset_index())
dpw_quant_cleaned["datetime_est"] = dpw_quant_cleaned["datetime_est"].dt.tz_localize(None)

dpw_quant_cleaned.head()
dpw_quant_cleaned.to_csv("cleaned quant data/dpw_cleaned.csv", index=False)

In [262]:
mjf_raw = (mjf_raw[["timestamp_local", "no2", "o3"]].rename(columns={"timestamp_local": "datetime_est"}))
mjf_raw["datetime_est"] = pd.to_datetime(mjf_raw["datetime_est"])

mjf_raw.head(30)

,datetime_est,no2,o3
0,2026-05-31 19:59:46+00:00,4.040,43.102
1,2026-05-31 19:58:46+00:00,3.379,45.299
2,2026-05-31 19:57:46+00:00,4.023,43.554
3,2026-05-31 19:56:46+00:00,3.572,44.712
4,2026-05-31 19:55:46+00:00,3.577,45.183
5,2026-05-31 19:54:46+00:00,3.063,45.568
6,2026-05-31 19:53:46+00:00,3.394,44.873
7,2026-05-31 19:52:46+00:00,3.384,44.853
8,2026-05-31 19:51:46+00:00,4.007,44.484
9,2026-05-31 19:50:46+00:00,3.396,45.788


In [263]:
mjf_quant_cleaned1 = (mjf_raw.set_index("datetime_est").resample("h").mean().reset_index())
mjf_quant_cleaned1["datetime_est"] = mjf_quant_cleaned1["datetime_est"].dt.tz_localize(None)

mjf_quant_cleaned1.head()

,datetime_est,no2,o3
0,2026-03-10 20:00:00,8.700633,50.574533
1,2026-03-10 21:00:00,10.013767,44.751100
2,2026-03-10 22:00:00,6.648233,46.229233
3,2026-03-10 23:00:00,6.199550,42.575500
4,2026-03-11 00:00:00,7.638633,37.286617


In [264]:
#recalibrated historical data from the MJF quant needs to be averaged and combined with recent historical data
mjf_recal = (mjf_recal[["timestamp_local", "no2", "o3"]].rename(columns={"timestamp_local": "datetime_est"}))

#some recalibrated timestamps have thousandths of seconds included for some reason. Remove those here
mjf_recal["datetime_est"] = (mjf_recal["datetime_est"].astype(str).str.replace(r"\.000$", "", regex=True))
mjf_recal["datetime_est"] = (mjf_recal["datetime_est"].astype(str).str.replace(r"\.500$", "", regex=True))

mjf_recal["datetime_est"] = pd.to_datetime(mjf_recal["datetime_est"])

mjf_recal.head(30)

,datetime_est,no2,o3
0,2025-08-28 14:16:12,-5.594733,235.480795
1,2025-08-28 14:17:12,-5.594733,214.030878
2,2025-08-28 14:18:12,-5.594733,195.068412
3,2025-08-28 14:19:12,-5.594732,-261.291336
4,2025-08-28 14:20:12,-5.594324,61.186989
5,2025-08-28 14:21:12,-5.593890,102.366522
6,2025-08-28 14:22:12,-5.591060,105.408751
7,2025-08-28 14:23:12,-5.565535,102.416938
8,2025-08-28 14:24:12,-5.089504,99.235495
9,2025-08-28 14:25:12,-4.854538,94.138863


In [265]:
mjf_quant_cleaned2 = (mjf_recal.set_index("datetime_est").resample("h").mean().reset_index())
mjf_quant_cleaned2["datetime_est"] = mjf_quant_cleaned2["datetime_est"].dt.tz_localize(None)

mjf_quant_cleaned2.head()

,datetime_est,no2,o3
0,2025-08-28 14:00:00,-1.744937,73.771410
1,2025-08-28 15:00:00,2.202533,57.932656
2,2025-08-28 16:00:00,2.432583,56.728456
3,2025-08-28 17:00:00,2.577775,56.368741
4,2025-08-28 18:00:00,3.062726,57.325889


In [266]:
mjf_quant_cleaned = pd.concat([mjf_quant_cleaned2, mjf_quant_cleaned1], ignore_index=True)

#can't merge directly because there are slight floating point differences between concentrations between recalibrated data and data from the portal
#ensure preference of web portal data here so no merge issues (tho almost no difference anyways)
mjf_quant_cleaned = (mjf_quant_cleaned.drop_duplicates(subset="datetime_est", keep="last").sort_values("datetime_est").reset_index(drop=True))

mjf_quant_cleaned.to_csv("cleaned quant data/mjf_cleaned.csv")

mjf_quant_cleaned.head()

,datetime_est,no2,o3
0,2025-08-28 14:00:00,-1.744937,73.771410
1,2025-08-28 15:00:00,2.202533,57.932656
2,2025-08-28 16:00:00,2.432583,56.728456
3,2025-08-28 17:00:00,2.577775,56.368741
4,2025-08-28 18:00:00,3.062726,57.325889


In [267]:
pema_raw = (pema_raw[["timestamp_local", "no2", "o3"]].rename(columns={"timestamp_local": "datetime_est"}))
pema_raw["datetime_est"] = pd.to_datetime(pema_raw["datetime_est"])

pema_raw.head(30)

,datetime_est,no2,o3
0,2026-05-31 19:59:22+00:00,16.724,21.874
1,2026-05-31 19:58:22+00:00,16.719,21.425
2,2026-05-31 19:57:22+00:00,17.292,22.222
3,2026-05-31 19:56:22+00:00,16.697,21.455
4,2026-05-31 19:55:22+00:00,16.609,23.275
5,2026-05-31 19:54:22+00:00,16.027,22.950
6,2026-05-31 19:53:22+00:00,17.187,21.327
7,2026-05-31 19:52:22+00:00,16.622,21.436
8,2026-05-31 19:51:22+00:00,18.398,21.029
9,2026-05-31 19:50:22+00:00,17.214,21.746


In [268]:
pema_quant_cleaned = (pema_raw.set_index("datetime_est").resample("h").mean().reset_index())
pema_quant_cleaned["datetime_est"] = pema_quant_cleaned["datetime_est"].dt.tz_localize(None)

pema_quant_cleaned.head()
pema_quant_cleaned.to_csv("cleaned quant data/pema_cleaned.csv", index=False)

In [269]:
pha_raw = (pha_raw[["timestamp_local", "no2", "o3"]].rename(columns={"timestamp_local": "datetime_est"}))
pha_raw["datetime_est"] = pd.to_datetime(pha_raw["datetime_est"])

pha_raw.head(30)

,datetime_est,no2,o3
0,2026-05-31 19:59:26+00:00,22.540,14.699
1,2026-05-31 19:58:26+00:00,23.036,15.473
2,2026-05-31 19:57:26+00:00,23.073,14.127
3,2026-05-31 19:56:26+00:00,21.967,15.263
4,2026-05-31 19:55:26+00:00,20.787,15.539
5,2026-05-31 19:54:26+00:00,23.023,15.466
6,2026-05-31 19:53:26+00:00,25.336,13.952
7,2026-05-31 19:52:26+00:00,24.520,14.228
8,2026-05-31 19:51:26+00:00,24.976,14.093
9,2026-05-31 19:50:26+00:00,23.648,14.956


In [270]:
pha_quant_cleaned = (pha_raw.set_index("datetime_est").resample("h").mean().reset_index())
pha_quant_cleaned["datetime_est"] = pha_quant_cleaned["datetime_est"].dt.tz_localize(None)

pha_quant_cleaned.head()
pha_quant_cleaned.to_csv("cleaned quant data/pha_cleaned.csv", index=False)